In [1]:
import os
import sys

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
import pyspark.sql.functions as f

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 06:09:46 WARN Utils: Your hostname, codespaces-f247df, resolves to a loopback address: 127.0.0.1; using 10.0.0.41 instead (on interface eth0)
26/03/05 06:09:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 06:09:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# !wget wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [4]:
print(spark.version)

4.1.1


In [5]:
try:
    yellow_202511_df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
    print("Success! Here is the data:")
    yellow_202511_df.show(5)
except Exception as e:
    print(f"Error: {e}")

Success! Here is the data:


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [6]:
yellow_202511_df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [6]:
schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True), 
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
    types.StructField('tpep_dropoff_datetime', types.TimestampNTZType(), True), 
    types.StructField('passenger_count', types.LongType(), True), 
    types.StructField('trip_distance', types.DoubleType(), True), 
    types.StructField('RatecodeID', types.LongType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('payment_type', types.LongType(), True), 
    types.StructField('fare_amount', types.DoubleType(), True), 
    types.StructField('extra', types.DoubleType(), True), 
    types.StructField('mta_tax', types.DoubleType(), True), 
    types.StructField('tip_amount', types.DoubleType(), True), 
    types.StructField('tolls_amount', types.DoubleType(), True), 
    types.StructField('improvement_surcharge', types.DoubleType(), True), 
    types.StructField('total_amount', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True), 
    types.StructField('Airport_fee', types.DoubleType(), True), 
    types.StructField('cbd_congestion_fee', types.DoubleType(), True)
    ]
)

In [9]:
df = yellow_202511_df.select(
    # Cast integers to LongType (BIGINT) and rename
    f.col("VendorID").cast(types.LongType()).alias("vendor_id"),
    f.col("PULocationID").cast(types.LongType()).alias("pickup_location_id"),
    f.col("DOLocationID").cast(types.LongType()).alias("dropoff_location_id"),
    
    # Rename datetimes, keep as TimestampNTZType
    f.col("tpep_pickup_datetime").alias("pickup_datetime"),
    f.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    
    # Keep existing types, clean up names for Postgres
    f.col("passenger_count"),
    f.col("trip_distance"),
    f.col("RatecodeID").alias("ratecode_id"), 
    f.col("store_and_fwd_flag"),
    f.col("payment_type"),
    f.col("fare_amount"),
    f.col("extra"),
    f.col("mta_tax"),
    f.col("tip_amount"),
    f.col("tolls_amount"),
    f.col("improvement_surcharge"),
    f.col("total_amount"),
    f.col("congestion_surcharge"),
    f.col("Airport_fee").alias("airport_fee"),
    f.col("cbd_congestion_fee")
)

In [11]:
df.printSchema()

root
 |-- vendor_id: long (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [12]:
df.show()

+---------+------------------+-------------------+-------------------+-------------------+---------------+-------------+-----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|vendor_id|pickup_location_id|dropoff_location_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+---------+------------------+-------------------+-------------------+-------------------+---------------+-------------+-----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|        7|                43|                186|2025-11-01 00:13:25|2025-11-01 00:13:25|    

In [26]:
repartitioned_df = df.repartition(4)
output_path = "partitioned_taxi_data"
repartitioned_df.write\
                .mode('overwrite')\
                .parquet(output_path)
print(f"Successfully saved to {output_path}/")

[Stage 47:=============================>                            (2 + 2) / 4]

Successfully saved to partitioned_taxi_data/


In [27]:
trips_cnt_11_15 = repartitioned_df.filter(f.col("pickup_datetime").cast("date") == "2025-11-15").count()

In [30]:
print(f"{trips_cnt_11_15} trips on 11/15.")

162604 trips on 11/15.


In [36]:
# Overwrite the variable to include the new column
repartitioned_df = repartitioned_df.withColumn(
    "duration_hours",
    (f.unix_timestamp(f.col("dropoff_datetime")) - f.unix_timestamp(f.col("pickup_datetime"))) / 3600.0
)

# Find the maximum duration using the updated DataFrame
repartitioned_df.select(f.max("duration_hours")).show()

[Stage 73:>                                                         (0 + 2) / 2]

+-------------------+
|max(duration_hours)|
+-------------------+
|  90.64666666666666|
+-------------------+



In [33]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-05 06:57:15--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.153, 52.85.39.65, 52.85.39.97, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-05 06:57:15 (1.03 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [44]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
zones_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("taxi_zone_lookup.csv")

In [45]:
zones_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [46]:
joined_df = repartitioned_df.join(
    zones_df,
    repartitioned_df["pickup_location_id"] == zones_df["LocationID"],
    how="left"
)
pickup_trips = joined_df.groupby("Zone").agg(f.count("*").alias("trips_num"))\
                        .orderBy(f.col("trips_num").asc())
pickup_trips.show(1,truncate=False)

[Stage 93:===========================================>              (3 + 1) / 4]

+---------------------------------------------+---------+
|Zone                                         |trips_num|
+---------------------------------------------+---------+
|Governor's Island/Ellis Island/Liberty Island|1        |
+---------------------------------------------+---------+
only showing top 1 row
